# 11 — Skill-Requirement Models: Evaluation & Summary Table

This notebook evaluates all 27 binary **skill-requirement models** trained in Chapter 1 and compiles their performance into a unified results table.  
Each model predicts whether a given skill group (e.g., *core_programming__basic*) is required for a job based on company attributes, role attributes, and all other skill indicators.

---

## **Goals of This Notebook**
- Load the processed dataset used for training skill models.
- For each of the 27 skill groups:
  - Recreate the train/test split.
  - Load the corresponding LightGBM model from disk.
  - Run the full evaluation pipeline (ROC AUC, PR AUC, Brier score, prevalence).
  - Extract the model's hyperparameters.
  - Extract full feature importances.
- Build a **single consolidated dataframe** where:
  - Each row corresponds to one skill model.
  - Columns contain performance metrics, model parameters, and feature-importance values aligned to their feature names.

---

## **What This Notebook Produces**
1. **A complete evaluation table** containing:
   - Train/test ROC AUC  
   - Train/test PR AUC  
   - Test Brier score  
   - Prevalence of the skill  
   - Hyperparameters used (learning rate, num leaves, n estimators, etc.)  
   - Variable-importance scores for every non-target feature  

2. **A reproducible evaluation pipeline** using the helper function  
   `evaluate_skill_model()`  
   which:
   - Loads the correct model from disk  
   - Applies consistent feature selection  
   - Computes metrics  
   - Returns both metrics and a tidy importance table  

3. **A modelling-ready summary dataframe** for downstream analysis:
   - Skill difficulty  
   - Predictability  
   - Feature drivers of each skill  
   - Prevalence–performance relationships  

---

## **Outcome**
This notebook provides the **complete diagnostic view** of all 27 skill models.  
It forms the analytical backbone for interpreting model behaviour, comparing skill predictability, and integrating the results into later chapters (probability matrix construction, recommender logic, and skill-value insights).

All output from this notebook remains internal to Chapter 1 and supports the next step: generating the **skill probability matrix** for all jobs.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import sys
from pathlib import Path
import pkgutil
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root


from src.job_intel.config import CH1_PROCESSED_SALARY_MODEL_PCA_DF

from src.job_intel.evaluation.skill_model_eval import evaluate_skill_model


In [ ]:
from sklearn.model_selection import train_test_split


df = pd.read_csv(CH1_PROCESSED_SALARY_MODEL_PCA_DF)


skills = ['core_programming__basic', 'core_programming__intermediate', 
            'core_programming__advanced', 'data_engineering_pipelines__basic',
            'data_engineering_pipelines__intermediate',
            'data_engineering_pipelines__advanced', 'ml_ai__basic',
            'ml_ai__intermediate', 'ml_ai__advanced', 'analytics_stats__basic',
            'analytics_stats__intermediate', 'analytics_stats__advanced',
            'bi_viz__basic', 'bi_viz__intermediate', 'bi_viz__advanced',
            'cloud__basic', 'cloud__intermediate', 'cloud__advanced',
            'db_storage__basic', 'db_storage__intermediate', 'db_storage__advanced',
            'productivity_workflow__basic', 'productivity_workflow__intermediate',
            'productivity_workflow__advanced', 'soft_skills__core',
            'soft_skills__leadership', 'domain_specific__none']

all_features = df[[
            # Company
            'size_code', 'sector_code', 'state_code','ownership_code',
            # Role
            'seniority_code', 'title_rich_code',
            # Skills
            'core_programming__basic', 'core_programming__intermediate', 
            'core_programming__advanced', 'data_engineering_pipelines__basic',
            'data_engineering_pipelines__intermediate',
            'data_engineering_pipelines__advanced', 'ml_ai__basic',
            'ml_ai__intermediate', 'ml_ai__advanced', 'analytics_stats__basic',
            'analytics_stats__intermediate', 'analytics_stats__advanced',
            'bi_viz__basic', 'bi_viz__intermediate', 'bi_viz__advanced',
            'cloud__basic', 'cloud__intermediate', 'cloud__advanced',
            'db_storage__basic', 'db_storage__intermediate', 'db_storage__advanced',
            'productivity_workflow__basic', 'productivity_workflow__intermediate',
            'productivity_workflow__advanced', 'soft_skills__core',
            'soft_skills__leadership', 'domain_specific__none',
            ]].copy()

cat_cols = [
    'size_code', 'sector_code', 'state_code', 'ownership_code', 'seniority_code', 'title_rich_code'
]
all_features[cat_cols] = all_features[cat_cols].astype('category')

cols = ['model',
        'pos_frac_train',
        'pos_frac_test',
        'roc_auc_train',
        'roc_auc_test',
        'pr_auc_train',
        'pr_auc_test',
        'brier_test',
        'learning_rate',
        'n_estimators',
        'num_leaves',
        'colsample_bytree',
        'subsample',
        'size_code', 'sector_code', 'state_code','ownership_code',
        'seniority_code', 'title_rich_code',
        'core_programming__basic', 'core_programming__intermediate', 
        'core_programming__advanced', 'data_engineering_pipelines__basic',
        'data_engineering_pipelines__intermediate',
        'data_engineering_pipelines__advanced', 'ml_ai__basic',
        'ml_ai__intermediate', 'ml_ai__advanced', 'analytics_stats__basic',
        'analytics_stats__intermediate', 'analytics_stats__advanced',
        'bi_viz__basic', 'bi_viz__intermediate', 'bi_viz__advanced',
        'cloud__basic', 'cloud__intermediate', 'cloud__advanced',
        'db_storage__basic', 'db_storage__intermediate', 'db_storage__advanced',
        'productivity_workflow__basic', 'productivity_workflow__intermediate',
        'productivity_workflow__advanced', 'soft_skills__core',
        'soft_skills__leadership', 'domain_specific__none',]

In [ ]:
results_df = pd.DataFrame()

for target in skills:

    y = df[target]
    X = all_features.drop(target, axis = 1)


    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )
    metrics, var_imp = evaluate_skill_model(
        response_name = target,
        X_train = X_train,
        y_train = y_train,
        X_test = X_test,
        y_test = y_test,
        model= None,
        show_plots = False)
    
    def build_eval_row(metrics: dict, var_imp: pd.DataFrame, all_cols: list[str]) -> pd.DataFrame:
        # Start with NaN for everything
        row = {col: np.nan for col in all_cols}

        # Model id
        
        row['model'] = target

        # fill metrics
        for k, v in metrics.items():
            if k in row:
                row[k] = v

        # fill variable importance
        # var_imp has columns: ["feature", "importance"]
        for _, r in var_imp.iterrows():
            feat = r["feature"]
            imp = r["importance"]
            if feat in row:
                row[feat] = imp
        

        # return as single-row DataFrame
        return pd.DataFrame([row])
    
    row_df = build_eval_row(metrics, var_imp, cols)
    results_df = pd.concat([results_df, row_df], ignore_index=True)



In [ ]:
results_df

### Save output

In [ ]:
from src.job_intel.config import PROCESSED_DATA_DIR
results_df.to_csv(PROCESSED_DATA_DIR / 'skill_model_evaluation_result.csv', index=False)

## Metrics visualisation and high-level interpretation

In [ ]:
results_df.describe()

In [ ]:
results_df[['model','pos_frac_train', 'pr_auc_test']].sort_values(by='pr_auc_test')

### Brier Score — Probability Calibration Quality

The Brier scores fall mostly between 0.05 and 0.12, with a small cluster of extremely low values near 0.00 for skills with very high prevalence. A lower Brier score indicates better calibration: predicted probabilities are close to the true empirical frequencies. Scores under ~0.12 reflect well-calibrated, sharply defined probabilities, meaning the model’s predicted skill probabilities can be interpreted directly and used safely in downstream applications such as job–job similarity and user–job matching. Even for rarer skills, the calibration penalty remains modest, confirming that the models produce stable, meaningful probability estimates.

### Interpretation

* 0.0 = perfect probabilistic predictions
* 0.25 = equivalent to always predicting 0.5
* over 0.25 = bad (worse than guessing)
* 1.0 = worst possible


In [ ]:
sns.histplot(
    data = results_df,
    x = 'brier_test',
    bins = 10
)

plt.title("Brier test")


### ROC AUC — Discrimination Ability

The ROC AUC distribution shows that almost all models achieve excellent discrimination, with most values clustered between 0.88 and 0.95, and only a few dipping toward 0.80. This means the models separate positive vs. negative skill occurrences very effectively across nearly all skill groups. A high ROC AUC confirms that LightGBM is reliably ranking jobs that truly require a skill higher than jobs that do not, independent of the skill’s prevalence. This is exactly what we want for a probability-based skill model.

### Interpretation

* 0.80–0.85 → good
* 0.85–0.90 → very good
* 0.90–0.95 → excellent
* 0.95+ → near-perfect discrimination

In [ ]:
sns.histplot(
    data = results_df,
    x = 'roc_auc_test',
    bins = 10
)

plt.title("ROC AUC")

### PR AUC — Performance Under Class Imbalance

The PR AUC values range widely, from ~0.40 for rare skills to ~0.95 for common or highly structured skills. This behaviour is expected: PR AUC is directly influenced by the base prevalence of each skill. Skills that occur frequently and have strong structural patterns in the job attributes achieve very high PR AUC, while sparse or weakly structured skills naturally show lower values. Importantly, even the lower PR AUC models still perform meaningfully above their prevalence baselines, demonstrating that the models are learning real signal rather than noise.



| PR AUC      | Interpretation                          |
| ----------- | --------------------------------------- |
| 0.30–0.50 | Rare or noisy skill; still acceptable   |
| 0.50–0.70 | Moderate performance; decent signal     |
| 0.70–0.85 | Strong model; good separation           |
| 0.85–0.95 | Very strong                             |
| 0.95+     | Almost perfect precision at high recall |

In [ ]:
sns.histplot(
    data = results_df,
    x = 'pr_auc_test',
    bins = 10
)

plt.title("PR AUC")

## Model parameters viz

In [ ]:
sns.histplot(
    data = results_df,
    x = 'learning_rate',
    bins = 10
)

plt.title("Learning rate")

In [ ]:
sns.histplot(
    data = results_df,
    x = 'n_estimators',
    bins = 10
)

plt.title("Estimators")

In [ ]:
var_importance = results_df.iloc[:,13:].fillna(0).corr()
var_importance